# Analysis - where in the decoder the request acts

Issue #9. Six adapters, one per layer, start at exactly zero and are trained in step 2. This zeros one at a time on
the saved generator and measures what it costs to write a held-out molecule once told its own odor tokens.

The numbers come from `python -m scripts.analysis.ablations`: 120 test molecules, the same seed for every condition.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data = json.loads((ROOT / "output/analysis/ablations.json").read_text())
intact = data["intact"]
layers = list(data["zeroed"])
rises = [data["zeroed"][layer] - intact for layer in layers]
print(f"{data['molecules']} molecules from {data['split']}, intact loss {intact:.4f}")

## 1. What each adapter is worth on its own

The rise in loss when that layer's adapter is set to zero, one layer at a time, with the other five intact.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.bar(layers, rises, color="#1F3864")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("layer whose adapter is zeroed")
ax.set_ylabel("rise in loss (nats per molecule token)")
ax.set_title("one adapter at a time")
for i, value in enumerate(rises):
    ax.text(i, value + 0.001, f"{value:+.3f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

print(pd.Series(rises, index=layers).round(4).to_string())
print(f"sum of the six alone: {sum(rises):.4f}")

## 2. All six at once

Zeroing every adapter should return the model to what it was before fine-tuning: a decoder that writes molecules
without being asked for anything.

In [ ]:
all_zero = data["all_zero"] - intact
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.bar(["one at a time, summed", "all six at once"], [sum(rises), all_zero], color=["#8FAADC", "#1F3864"])
ax.set_ylabel("rise in loss (nats per molecule token)")
for i, value in enumerate([sum(rises), all_zero]):
    ax.text(i, value + 0.004, f"{value:+.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print(f"intact {intact:.4f}, all six zeroed {data['all_zero']:.4f}, rise {all_zero:+.4f}")

## Reading.

The last adapter carries the most by a wide margin: zeroing it alone costs +0.065 nats, against +0.008 to +0.023
for the other five. The first two carry almost nothing on their own. That matches the norms, which rise from 4.3 at
layer one to 8.9 at layer six, and the layer decay that produced them: the earlier layers learn at a fraction of the
later ones' rate.

The six do not add up. Zeroing them one at a time and summing gives +0.138, while zeroing all six at once gives
+0.227, and the model with all six gone reaches 0.9666, against 0.9239 for an empty request on the same split: the
adapters explain most of the request effect, not all of it. The adapters
substitute for one another: with five gone, the sixth still carries part of the request, and only the removal of all
of them returns the model to the pretrained behaviour.

## Verdict

The request is carried by what is added at every layer, concentrated at the top. This is the causal counterpart of
two other observations: the odor embeddings move only 3.1 percent in fine-tuning, and zeroing all six adapters
recovers the no-request loss measured in the evaluation. Three measurements, one story.
